# Hiver SDE Intern Take-Home Assignment

## AI Customer Support Agent — AmericanAir

This notebook contains the cleaned implementation and evaluation workflow.

### Pipeline

1. Dataset preparation
2. Brand selection
3. Customer-support pair construction
4. Intent discovery and classification
5. Historical evidence retrieval
6. Grounded reply generation
7. Safety and escalation decision
8. Evaluation and failure analysis

Manual annotation and temporary debugging cells from the development notebook are excluded from this submission version. Their final results are preserved in the `evaluation/` directory.


In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib seaborn tqdm

In [ ]:
import pandas as pd
import numpy as np
import os
import re
from collections import Counter
from tqdm.auto import tqdm

print("Environment ready!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATA_PATH = "/content/drive/MyDrive/Hiver SDE/Dataset/twcs/twcs.csv"

print(os.path.exists(DATA_PATH))

In [ ]:
sample = pd.read_csv(
    DATA_PATH,
    nrows=10,
    low_memory=False
)

print("Columns:")
print(sample.columns.tolist())

print("\nFirst 5 rows:")
display(sample.head())

In [ ]:
total_rows = 0

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["tweet_id"],
    chunksize=200_000,
    low_memory=False
):
    total_rows += len(chunk)

print(f"Total rows: {total_rows:,}")

In [ ]:
# Step 2: Brand-wise dataset profiling

import pandas as pd
from collections import Counter

brand_stats = Counter()

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["author_id", "inbound"],
    chunksize=200_000,
    low_memory=False
):
    # Outbound tweets are generally replies sent by support accounts.
    outbound = chunk[chunk["inbound"] == False]

    brand_stats.update(
        outbound["author_id"].dropna().astype(str)
    )

# Convert to DataFrame
brand_df = pd.DataFrame(
    brand_stats.items(),
    columns=["author_id", "outbound_tweets"]
)

brand_df = brand_df.sort_values(
    "outbound_tweets",
    ascending=False
).reset_index(drop=True)

display(brand_df.head(30))

In [ ]:
# Step 2B: Compare inbound and outbound activity

top_authors = brand_df.head(30)["author_id"].tolist()

stats = []

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["author_id", "inbound"],
    chunksize=200_000,
    low_memory=False
):
    chunk["author_id"] = chunk["author_id"].astype(str)

    selected = chunk[
        chunk["author_id"].isin(top_authors)
    ]

    if len(selected) == 0:
        continue

    grouped = (
        selected
        .groupby(["author_id", "inbound"])
        .size()
        .unstack(fill_value=0)
    )

    for author in grouped.index:
        inbound_count = grouped.loc[author].get(True, 0)
        outbound_count = grouped.loc[author].get(False, 0)

        stats.append({
            "author_id": author,
            "inbound": inbound_count,
            "outbound": outbound_count
        })

brand_activity = pd.DataFrame(stats)

brand_activity = (
    brand_activity
    .groupby("author_id", as_index=False)
    .sum()
)

brand_activity["total"] = (
    brand_activity["inbound"]
    + brand_activity["outbound"]
)

brand_activity = brand_activity.sort_values(
    "total",
    ascending=False
)

display(brand_activity)

In [ ]:
# Step 2C — Correct customer -> support conversation analysis

import pandas as pd
from collections import Counter

candidate_brands = [
    "AmazonHelp",
    "AppleSupport",
    "Uber_Support",
    "SpotifyCares",
    "Delta",
    "Tesco",
    "AmericanAir",
    "TMobileHelp",
    "comcastcares",
    "British_Airways"
]

brand_set = set(candidate_brands)

stats = {
    brand: {
        "support_tweets": 0,
        "support_replies_to_customers": 0,
        "unique_customer_messages_replied_to": set()
    }
    for brand in candidate_brands
}

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=[
        "tweet_id",
        "author_id",
        "inbound",
        "in_response_to_tweet_id"
    ],
    chunksize=200_000,
    low_memory=False
):

    chunk["author_id"] = chunk["author_id"].astype(str)

    # Brand/support tweets
    support = chunk[
        (chunk["author_id"].isin(brand_set)) &
        (chunk["inbound"] == False)
    ].copy()

    for brand in support["author_id"].unique():

        brand_rows = support[
            support["author_id"] == brand
        ]

        stats[brand]["support_tweets"] += len(brand_rows)

        # A support tweet with a parent means
        # the support agent is replying to another tweet.
        replied = brand_rows[
            brand_rows["in_response_to_tweet_id"].notna()
        ]

        stats[brand]["support_replies_to_customers"] += len(replied)

        stats[brand]["unique_customer_messages_replied_to"].update(
            replied["in_response_to_tweet_id"]
            .dropna()
            .astype(str)
        )

results = []

for brand, s in stats.items():

    results.append({
        "brand": brand,
        "support_tweets": s["support_tweets"],
        "support_replies": s["support_replies_to_customers"],
        "unique_customer_threads": len(
            s["unique_customer_messages_replied_to"]
        )
    })

conversation_df = pd.DataFrame(results)

conversation_df["reply_coverage"] = (
    conversation_df["support_replies"]
    / conversation_df["support_tweets"]
)

conversation_df = conversation_df.sort_values(
    "support_replies",
    ascending=False
)

display(conversation_df)

In [ ]:
# Step 2D — Inspect real conversations for candidate brands

candidate_brands = [
    "AmazonHelp",
    "AppleSupport",
    "Uber_Support",
    "SpotifyCares",
    "Delta",
    "Tesco",
    "AmericanAir",
    "TMobileHelp",
    "comcastcares",
    "British_Airways"
]

sample_rows = []

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=[
        "tweet_id",
        "author_id",
        "inbound",
        "created_at",
        "text",
        "response_tweet_id",
        "in_response_to_tweet_id"
    ],
    chunksize=200_000,
    low_memory=False
):

    matched = chunk[
        chunk["author_id"].astype(str).isin(candidate_brands)
    ].copy()

    if len(matched) > 0:
        sample_rows.append(matched)

    # We only need a manageable amount for inspection
    if sum(len(x) for x in sample_rows) >= 20_000:
        break

candidate_sample = pd.concat(sample_rows, ignore_index=True)

print("Rows collected:", len(candidate_sample))

display(
    candidate_sample[
        [
            "author_id",
            "inbound",
            "text",
            "response_tweet_id",
            "in_response_to_tweet_id"
        ]
    ].head(30)
)

In [ ]:
# Count actual support replies with a parent tweet

candidate_sample["has_parent"] = (
    candidate_sample["in_response_to_tweet_id"].notna()
)

summary = (
    candidate_sample[
        (candidate_sample["inbound"] == False)
    ]
    .groupby("author_id")
    .agg(
        support_tweets=("tweet_id", "count"),
        replies_with_parent=("has_parent", "sum")
    )
    .reset_index()
)

summary["parent_rate"] = (
    summary["replies_with_parent"] /
    summary["support_tweets"]
)

summary = summary.sort_values(
    "support_tweets",
    ascending=False
)

display(summary)

In [ ]:
# Show actual support replies that have parent tweets

examples = candidate_sample[
    (candidate_sample["inbound"] == False) &
    (candidate_sample["in_response_to_tweet_id"].notna())
][
    [
        "author_id",
        "text",
        "in_response_to_tweet_id"
    ]
].head(20)

display(examples)

In [ ]:
# Step 2E — Final candidate comparison
# We sample real customer-support pairs for each candidate brand.

import pandas as pd
from collections import defaultdict

candidate_brands = [
    "AmazonHelp",
    "AppleSupport",
    "Uber_Support",
    "SpotifyCares",
    "Delta",
    "Tesco",
    "AmericanAir",
    "TMobileHelp",
    "comcastcares",
    "British_Airways"
]

brand_set = set(candidate_brands)

# Store support tweets and the customer tweet IDs they respond to
support_pairs = defaultdict(list)

MAX_PAIRS_PER_BRAND = 1000

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=[
        "tweet_id",
        "author_id",
        "inbound",
        "text",
        "in_response_to_tweet_id"
    ],
    chunksize=200_000,
    low_memory=False
):

    chunk["author_id"] = chunk["author_id"].astype(str)

    support = chunk[
        (chunk["author_id"].isin(brand_set)) &
        (chunk["inbound"] == False) &
        (chunk["in_response_to_tweet_id"].notna())
    ]

    for brand in candidate_brands:

        rows = support[
            support["author_id"] == brand
        ]

        remaining = MAX_PAIRS_PER_BRAND - len(support_pairs[brand])

        if remaining <= 0:
            continue

        rows = rows.head(remaining)

        for _, row in rows.iterrows():
            support_pairs[brand].append({
                "support_tweet_id": str(row["tweet_id"]),
                "customer_tweet_id": str(
                    row["in_response_to_tweet_id"]
                ),
                "support_text": str(row["text"])
            })

    # Stop once all brands have enough examples
    if all(
        len(support_pairs[b]) >= MAX_PAIRS_PER_BRAND
        for b in candidate_brands
    ):
        break

print("Pairs collected:")

for brand in candidate_brands:
    print(
        f"{brand:20} "
        f"{len(support_pairs[brand]):,}"
    )

In [ ]:
# STEP 2F — Robust customer-message retrieval

def normalize_id(x):
    if pd.isna(x):
        return None

    s = str(x).strip()

    # Convert values like 272.0 -> 272
    if s.endswith(".0"):
        s = s[:-2]

    return s


# Normalize customer tweet IDs collected in Step 2E
for brand in candidate_brands:
    for pair in support_pairs[brand]:
        pair["customer_tweet_id"] = normalize_id(pair["customer_tweet_id"])


# Collect all required customer tweet IDs
customer_ids = set()

for brand in candidate_brands:
    for pair in support_pairs[brand]:
        cid = pair["customer_tweet_id"]
        if cid:
            customer_ids.add(cid)

print("Customer IDs to retrieve:", len(customer_ids))


# Retrieve customer messages from full dataset
customer_messages = {}

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=["tweet_id", "author_id", "inbound", "text", "created_at"],
    dtype={
        "tweet_id": "string",
        "author_id": "string"
    },
    chunksize=200_000,
    low_memory=False
):

    chunk["tweet_id"] = chunk["tweet_id"].map(normalize_id)

    matched = chunk[chunk["tweet_id"].isin(customer_ids)]

    for _, row in matched.iterrows():

        tweet_id = row["tweet_id"]

        customer_messages[tweet_id] = {
            "author_id": row["author_id"],
            "text": row["text"],
            "created_at": row["created_at"]
        }

print("Customer messages retrieved:", len(customer_messages))

In [ ]:
# STEP 2G — Compare customer-message quality across candidate brands

results = []

for brand in candidate_brands:

    pairs = support_pairs[brand]

    customer_texts = []

    for pair in pairs:

        customer_id = normalize_id(pair["customer_tweet_id"])

        if customer_id in customer_messages:

            text = customer_messages[customer_id]["text"]

            if pd.notna(text) and str(text).strip():
                customer_texts.append(str(text).strip())

    if customer_texts:

        lengths = [len(text) for text in customer_texts]

        non_empty = sum(len(text.strip()) > 0 for text in customer_texts)

        short_messages = sum(
            len(text.strip()) < 20
            for text in customer_texts
        )

        unique_messages = len(set(customer_texts))

        results.append({
            "brand": brand,
            "support_customer_pairs": len(pairs),
            "customer_text_found": len(customer_texts),
            "unique_customer_messages": unique_messages,
            "non_empty_rate": round(
                non_empty / len(customer_texts), 3
            ),
            "short_message_rate": round(
                short_messages / len(customer_texts), 3
            ),
            "median_customer_length": int(
                pd.Series(lengths).median()
            )
        })


final_brand_df = pd.DataFrame(results)

display(
    final_brand_df.sort_values(
        "customer_text_found",
        ascending=False
    )
)

In [ ]:
# STEP 3A — Extract AmericanAir customer → support pairs

TARGET_BRAND = "AmericanAir"

american_pairs = []

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=[
        "tweet_id",
        "author_id",
        "inbound",
        "created_at",
        "text",
        "response_tweet_id",
        "in_response_to_tweet_id"
    ],
    dtype={
        "tweet_id": "string",
        "author_id": "string",
        "in_response_to_tweet_id": "string"
    },
    chunksize=200_000,
    low_memory=False
):

    chunk["tweet_id"] = chunk["tweet_id"].map(normalize_id)
    chunk["in_response_to_tweet_id"] = (
        chunk["in_response_to_tweet_id"].map(normalize_id)
    )

    # AmericanAir support replies
    support = chunk[
        (chunk["author_id"] == TARGET_BRAND) &
        (chunk["inbound"] == False) &
        (chunk["in_response_to_tweet_id"].notna())
    ]

    for _, row in support.iterrows():

        american_pairs.append({
            "support_tweet_id": row["tweet_id"],
            "customer_tweet_id": row["in_response_to_tweet_id"],
            "support_text": row["text"],
            "support_created_at": row["created_at"]
        })


american_pairs_df = pd.DataFrame(american_pairs)

print("AmericanAir support replies:", len(american_pairs_df))

display(american_pairs_df.head(10))

In [ ]:
# STEP 3B — Retrieve customer messages for AmericanAir

american_customer_ids = set(
    american_pairs_df["customer_tweet_id"]
    .dropna()
    .map(normalize_id)
)

print("Customer messages required:", len(american_customer_ids))


american_customers = {}

for chunk in pd.read_csv(
    DATA_PATH,
    usecols=[
        "tweet_id",
        "author_id",
        "inbound",
        "created_at",
        "text",
        "response_tweet_id",
        "in_response_to_tweet_id"
    ],
    dtype={
        "tweet_id": "string",
        "author_id": "string"
    },
    chunksize=200_000,
    low_memory=False
):

    chunk["tweet_id"] = chunk["tweet_id"].map(normalize_id)

    matched = chunk[
        chunk["tweet_id"].isin(american_customer_ids)
    ]

    for _, row in matched.iterrows():

        american_customers[row["tweet_id"]] = {
            "customer_tweet_id": row["tweet_id"],
            "customer_author_id": row["author_id"],
            "customer_text": row["text"],
            "customer_created_at": row["created_at"],
            "response_tweet_id": row["response_tweet_id"]
        }


print(
    "Customer messages retrieved:",
    len(american_customers)
)

In [ ]:
# STEP 3C — Create usable customer → support records

records = []

for _, row in american_pairs_df.iterrows():

    customer_id = normalize_id(row["customer_tweet_id"])

    if customer_id not in american_customers:
        continue

    customer = american_customers[customer_id]

    customer_text = str(customer["customer_text"]).strip()
    support_text = str(row["support_text"]).strip()

    if not customer_text or customer_text.lower() == "nan":
        continue

    if not support_text or support_text.lower() == "nan":
        continue

    records.append({
        "customer_tweet_id": customer_id,
        "support_tweet_id": row["support_tweet_id"],
        "customer_text": customer_text,
        "support_text": support_text,
        "customer_created_at": customer["customer_created_at"],
        "support_created_at": row["support_created_at"]
    })


american_df = pd.DataFrame(records)

print("Final usable customer → support pairs:", len(american_df))

display(american_df.head(10))

In [ ]:
# STEP 4A — Prepare AmericanAir customer messages

intent_df = american_df.copy()

# Basic text cleanup
intent_df["customer_text"] = (
    intent_df["customer_text"]
    .fillna("")
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Remove empty messages
intent_df = intent_df[
    intent_df["customer_text"].str.len() > 0
].copy()

print("Total usable pairs:", len(intent_df))
print(
    "Unique customer tweets:",
    intent_df["customer_tweet_id"].nunique()
)
print(
    "Unique customer messages:",
    intent_df["customer_text"].nunique()
)

display(
    intent_df[
        ["customer_tweet_id", "customer_text", "support_text"]
    ].head(10)
)

In [ ]:
# STEP 4B — Inspect common customer terms

from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import re

texts = intent_df["customer_text"].tolist()

vectorizer_preview = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.95
)

X_preview = vectorizer_preview.fit_transform(texts)

terms = vectorizer_preview.get_feature_names_out()

term_counts = X_preview.sum(axis=0).A1

top_terms = (
    pd.DataFrame({
        "term": terms,
        "tfidf_score": term_counts
    })
    .sort_values("tfidf_score", ascending=False)
    .head(30)
)

display(top_terms)

In [ ]:
# STEP 4C — TF-IDF + clustering for intent discovery

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import MiniBatchKMeans

# Remove common Twitter-specific noise
intent_df["clean_text"] = (
    intent_df["customer_text"]
    .str.replace(r"https?://\S+", " ", regex=True)
    .str.replace(r"@\w+", " ", regex=True)
    .str.replace(r"\bamp\b", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Remove very short messages
intent_df = intent_df[
    intent_df["clean_text"].str.len() >= 15
].copy()

print("Messages used for intent discovery:", len(intent_df))


vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.95,
    max_features=30000
)

X = vectorizer.fit_transform(
    intent_df["clean_text"]
)

print("TF-IDF matrix shape:", X.shape)


# Candidate intent clusters
N_CLUSTERS = 12

kmeans = MiniBatchKMeans(
    n_clusters=N_CLUSTERS,
    random_state=42,
    batch_size=2048,
    n_init=10
)

intent_df["cluster"] = kmeans.fit_predict(X)

print("\nCluster distribution:")

display(
    intent_df["cluster"]
    .value_counts()
    .sort_index()
)

In [ ]:
# STEP 4D — Inspect representative examples from each cluster

import numpy as np

for cluster_id in range(N_CLUSTERS):

    cluster_mask = (
        intent_df["cluster"] == cluster_id
    )

    cluster_indices = np.where(
        cluster_mask.values
    )[0]

    if len(cluster_indices) == 0:
        continue

    cluster_matrix = X[cluster_indices]

    centroid = kmeans.cluster_centers_[cluster_id]

    similarities = np.asarray(
        cluster_matrix @ centroid
    ).ravel()

    top_positions = similarities.argsort()[-10:][::-1]

    print("\n" + "=" * 90)
    print(
        f"CLUSTER {cluster_id} "
        f"(n={len(cluster_indices)})"
    )
    print("=" * 90)

    for rank, position in enumerate(top_positions, 1):

        original_index = cluster_indices[position]

        print(
            f"{rank}. "
            f"{intent_df.iloc[original_index]['customer_text']}"
        )

In [ ]:
# STEP 5A — Scan candidate support topics

topic_patterns = {
    "baggage": r"\b(bag|bags|baggage|luggage|suitcase|carry[- ]?on)\b",
    "seat": r"\b(seat|seating|assigned seat|seat assignment)\b",
    "flight_delay": r"\b(delay|delayed|late|running late)\b",
    "flight_cancel": r"\b(cancel|cancelled|canceled|cancellation)\b",
    "booking": r"\b(book|booking|reservation|reserve)\b",
    "refund": r"\b(refund|reimburse|reimbursement|money back)\b",
    "rebooking": r"\b(rebook|rebooking|change my flight|change flight)\b",
    "check_in": r"\b(check[- ]?in|checkin)\b",
    "boarding": r"\b(board|boarding|boarding pass)\b",
    "gate": r"\b(gate|gate agent)\b",
    "payment": r"\b(charge|charged|payment|pay|fee|fees)\b",
    "miles": r"\b(miles|mileage|aadvantage|points)\b",
    "wifi": r"\b(wifi|wi-fi|internet)\b",
    "entertainment": r"\b(tv|movie|movies|entertainment|screen)\b",
    "airport": r"\b(airport|terminal)\b",
    "customer_service": r"\b(customer service|phone number|representative|agent)\b",
    "accessibility": r"\b(wheelchair|disabled|disability|accessible|accessibility)\b",
    "lost_item": r"\b(lost|missing).{0,30}\b(item|phone|wallet|laptop|belongings)\b"
}

topic_results = []

texts = intent_df["customer_text"].fillna("").astype(str)

for topic, pattern in topic_patterns.items():

    mask = texts.str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    )

    count = mask.sum()

    topic_results.append({
        "topic": topic,
        "matching_messages": int(count),
        "percentage": round(
            100 * count / len(intent_df),
            2
        )
    })


topic_df = (
    pd.DataFrame(topic_results)
    .sort_values(
        "matching_messages",
        ascending=False
    )
)

display(topic_df)

In [ ]:
# STEP 5B — Show representative examples for candidate topics

for topic, pattern in topic_patterns.items():

    mask = texts.str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    )

    examples = intent_df.loc[
        mask,
        ["customer_text", "support_text"]
    ].drop_duplicates(
        subset=["customer_text"]
    ).head(5)

    if len(examples) == 0:
        continue

    print("\n" + "=" * 90)
    print(f"TOPIC: {topic}")
    print("=" * 90)

    for i, (_, row) in enumerate(examples.iterrows(), 1):

        print(f"\n{i}. CUSTOMER:")
        print(row["customer_text"])

        print("   SUPPORT:")
        print(row["support_text"])

In [ ]:
# STEP 6A — Build historical resolution corpus

resolution_df = american_df.copy()

resolution_df["customer_text"] = (
    resolution_df["customer_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

resolution_df["support_text"] = (
    resolution_df["support_text"]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Remove empty records
resolution_df = resolution_df[
    (resolution_df["customer_text"].str.len() > 0) &
    (resolution_df["support_text"].str.len() > 0)
].copy()

# Remove exact duplicate customer-support pairs
resolution_df = resolution_df.drop_duplicates(
    subset=["customer_text", "support_text"]
).reset_index(drop=True)

print("Resolution corpus size:", len(resolution_df))

display(
    resolution_df[
        [
            "customer_tweet_id",
            "support_tweet_id",
            "customer_text",
            "support_text"
        ]
    ].head(10)
)

In [ ]:
# STEP 6B — TF-IDF retrieval baseline

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

retrieval_vectorizer = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_features=50000
)

retrieval_matrix = retrieval_vectorizer.fit_transform(
    resolution_df["customer_text"]
)

print("Retrieval matrix shape:", retrieval_matrix.shape)

In [ ]:
# STEP 6C — Historical similar-case retrieval

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_similar_cases(
    query,
    top_k=5,
    min_similarity=0.0
):
    """
    Retrieve historically similar customer-support cases.
    """

    query_vector = retrieval_vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vector,
        retrieval_matrix
    ).ravel()

    # Get top candidates efficiently
    top_indices = np.argsort(
        similarities
    )[-top_k:][::-1]

    results = resolution_df.iloc[
        top_indices
    ].copy()

    results["similarity"] = similarities[
        top_indices
    ]

    results = results[
        results["similarity"] >= min_similarity
    ]

    return results[
        [
            "customer_text",
            "support_text",
            "similarity"
        ]
    ].reset_index(drop=True)

In [ ]:
# STEP 6D — Test retrieval quality

test_queries = {
    "flight_delay":
        "My flight is delayed by three hours. What is happening?",

    "baggage":
        "My checked bag has not arrived. Where is my luggage?",

    "seat":
        "I cannot select my seat for my upcoming flight.",

    "booking":
        "I am trying to book a flight but the website is giving me an error.",

    "refund":
        "My flight was cancelled and I need a refund."
}


for intent, query in test_queries.items():

    print("\n" + "=" * 100)
    print(f"QUERY — {intent.upper()}")
    print("=" * 100)
    print(query)

    results = retrieve_similar_cases(
        query,
        top_k=5
    )

    for i, row in results.iterrows():

        print(f"\n--- Result {i+1} ---")
        print(
            f"Similarity: "
            f"{row['similarity']:.3f}"
        )

        print(
            "Customer:",
            row["customer_text"]
        )

        print(
            "Historical support:",
            row["support_text"]
        )

In [ ]:
# STEP 7A — Inspect temporal coverage

resolution_df["customer_created_at"] = pd.to_datetime(
    resolution_df["customer_created_at"],
    errors="coerce",
    utc=True
)

print("Earliest customer message:")
print(resolution_df["customer_created_at"].min())

print("\nLatest customer message:")
print(resolution_df["customer_created_at"].max())

print("\nMissing timestamps:")
print(
    resolution_df["customer_created_at"].isna().sum()
)

In [ ]:
# Step 7B — Chronological train/evaluation split

# Sort chronologically
resolution_df = resolution_df.sort_values("customer_created_at").reset_index(drop=True)

# Use the 80th percentile timestamp as the cutoff
cutoff_time = resolution_df["customer_created_at"].quantile(0.80)

train_df = resolution_df[
    resolution_df["customer_created_at"] < cutoff_time
].copy()

eval_pool_df = resolution_df[
    resolution_df["customer_created_at"] >= cutoff_time
].copy()

print("Temporal cutoff:")
print(cutoff_time)

print("\nTraining / historical evidence corpus:")
print("Rows:", len(train_df))
print("Earliest:", train_df["customer_created_at"].min())
print("Latest:", train_df["customer_created_at"].max())

print("\nEvaluation pool:")
print("Rows:", len(eval_pool_df))
print("Earliest:", eval_pool_df["customer_created_at"].min())
print("Latest:", eval_pool_df["customer_created_at"].max())

print("\nTrain proportion:", round(len(train_df) / len(resolution_df), 4))
print("Eval proportion:", round(len(eval_pool_df) / len(resolution_df), 4))

In [ ]:
# Step 8A — Candidate golden-set sampling buckets

golden_source = eval_pool_df.copy()

# Use cleaned customer text for matching
golden_source["clean_text"] = (
    golden_source["customer_text"]
    .fillna("")
    .astype(str)
    .str.replace(r"https?://\S+", " ", regex=True)
    .str.replace(r"@\w+", " ", regex=True)
    .str.replace(r"\bamp\b", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Keyword-assisted candidate buckets.
# These are ONLY for sampling; they are NOT ground-truth labels.

intent_patterns = {
    "flight_delay_status": r"\b(delay|delayed|late|lateness|eta|status|on time|running late)\b",

    "cancellation_schedule": r"\b(cancel|cancelled|canceled|cancellation|schedule change)\b",

    "rebooking_itinerary": r"\b(rebook|rebooking|rebooked|connection|missed flight|missed connection|alternate flight)\b",

    "baggage": r"\b(bag|bags|baggage|luggage|suitcase|carry.?on|checked bag|lost bag|missing bag|damaged bag)\b",

    "seat": r"\b(seat|seating|seatback|window seat|aisle|middle seat|seat assignment)\b",

    "booking": r"\b(book|booking|reservation|reserve|ticket|purchase|website|error|503)\b",

    "checkin_boarding_gate": r"\b(check.?in|checkin|boarding pass|board|boarding|gate)\b",

    "payment_fees_refund": r"\b(payment|paid|charge|charged|fee|fees|fare|refund|refunds|reimburse|reimbursement)\b",

    "loyalty": r"\b(miles|mileage|aadvantage|advantage|points|loyalty|elite|card benefits)\b",

    "onboard_services": r"\b(wifi|wi-fi|internet|entertainment|screen|movie|tv|seatback)\b",

    "accessibility": r"\b(wheelchair|accessible|accessibility|disability|disabled|service animal|special assistance)\b",

    "lost_found": r"\b(lost|left behind|lost and found|found item|laptop|phone|wallet)\b",
}

for intent, pattern in intent_patterns.items():
    golden_source[intent] = golden_source["clean_text"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    )

print("Evaluation pool:", len(golden_source))
print("\nCandidate counts by sampling bucket:")

for intent in intent_patterns:
    print(f"{intent:25s}: {golden_source[intent].sum():5d}")

In [ ]:
# Step 8B — Create a stratified 200-case golden evaluation candidate set

import numpy as np

np.random.seed(42)

sample_sizes = {
    "flight_delay_status": 25,
    "cancellation_schedule": 20,
    "rebooking_itinerary": 15,
    "baggage": 25,
    "seat": 18,
    "booking": 18,
    "checkin_boarding_gate": 25,
    "payment_fees_refund": 20,
    "loyalty": 12,
    "onboard_services": 10,
    "accessibility": 6,
    "lost_found": 6,
}

selected_parts = []

for bucket, n in sample_sizes.items():

    candidates = golden_source[golden_source[bucket]].copy()

    # Remove rows already selected through another bucket
    if selected_parts:
        already_selected = pd.concat(selected_parts)["customer_tweet_id"].unique()
        candidates = candidates[
            ~candidates["customer_tweet_id"].isin(already_selected)
        ]

    n_available = len(candidates)

    if n_available < n:
        print(
            f"WARNING: {bucket} only has {n_available} available; "
            f"requested {n}"
        )
        n = n_available

    sampled = candidates.sample(n=n, random_state=42)

    # Store the sampling bucket only as metadata.
    sampled["sampling_bucket"] = bucket

    selected_parts.append(sampled)


# Combine all targeted samples
golden_candidates = pd.concat(
    selected_parts,
    ignore_index=True
)

print("Targeted cases selected:", len(golden_candidates))

In [ ]:
# Step 8B continued — Add ambiguous/general cases

# Cases that do not cleanly fall into a single keyword bucket,
# or contain overlapping signals.

bucket_columns = list(intent_patterns.keys())

golden_source["bucket_match_count"] = golden_source[bucket_columns].sum(axis=1)

# Prefer cases with:
# 0 matching buckets = genuinely underspecified/general
# multiple matching buckets = ambiguous/overlapping
ambiguous_pool = golden_source[
    (golden_source["bucket_match_count"] == 0) |
    (golden_source["bucket_match_count"] >= 2)
].copy()

# Remove cases already selected
already_selected = golden_candidates["customer_tweet_id"].unique()

ambiguous_pool = ambiguous_pool[
    ~ambiguous_pool["customer_tweet_id"].isin(already_selected)
]

ambiguous_cases = ambiguous_pool.sample(
    n=min(20, len(ambiguous_pool)),
    random_state=42
).copy()

ambiguous_cases["sampling_bucket"] = "ambiguous_general"

golden_candidates = pd.concat(
    [golden_candidates, ambiguous_cases],
    ignore_index=True
)

print("Final candidate count:", len(golden_candidates))
print("\nSampling distribution:")
print(golden_candidates["sampling_bucket"].value_counts())

In [ ]:
# Step 8C — Prepare the 220-case human annotation sheet

golden_eval = golden_candidates[
    [
        "customer_tweet_id",
        "customer_created_at",
        "customer_text",
        "sampling_bucket"
    ]
].copy()

# Shuffle cases so similar intents are not grouped together
golden_eval = golden_eval.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Stable evaluation ID
golden_eval.insert(
    0,
    "eval_id",
    range(1, len(golden_eval) + 1)
)

# Human annotation fields
golden_eval["human_intent"] = ""
golden_eval["safe_to_auto_handle"] = ""
golden_eval["evidence_sufficient"] = ""
golden_eval["escalation_reason"] = ""
golden_eval["expected_reply_points"] = ""

print("Golden evaluation cases:", len(golden_eval))

display(golden_eval.head(10))

In [ ]:
# Step 8D — Export golden evaluation set

golden_path = "/content/golden_eval_americanair_220.csv"

golden_eval.to_csv(
    golden_path,
    index=False
)

print("Golden evaluation file saved:")
print(golden_path)

In [ ]:
import pandas as pd
import json

# Load our final golden evaluation set
golden_eval = pd.read_csv("/content/golden_eval_americanair_220_FINAL.csv")

print("Golden set loaded:", len(golden_eval))
print(golden_eval.columns.tolist())

In [ ]:
# Check the historical training corpus
print("Historical corpus:", len(train_df))
print("Columns:", train_df.columns.tolist())

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Use ONLY historical training data
eval_retrieval_df = train_df[
    ["customer_tweet_id",
     "support_tweet_id",
     "customer_text",
     "support_text",
     "customer_created_at",
     "support_created_at"]
].dropna(subset=["customer_text", "support_text"]).copy()

eval_retrieval_df["customer_text"] = (
    eval_retrieval_df["customer_text"]
    .astype(str)
    .str.strip()
)

eval_retrieval_df["support_text"] = (
    eval_retrieval_df["support_text"]
    .astype(str)
    .str.strip()
)

# Build retrieval index ONLY on historical data
eval_vectorizer = TfidfVectorizer(
    max_features=60000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

eval_retrieval_matrix = eval_vectorizer.fit_transform(
    eval_retrieval_df["customer_text"]
)

print("Temporal-safe retrieval corpus:", len(eval_retrieval_df))
print("Retrieval matrix:", eval_retrieval_matrix.shape)

In [ ]:
def retrieve_historical_cases(query, top_k=3):
    """
    Retrieve cases only from the historical training period.
    """

    query_vector = eval_vectorizer.transform([query])

    similarities = cosine_similarity(
        query_vector,
        eval_retrieval_matrix
    ).ravel()

    top_indices = np.argsort(similarities)[-top_k:][::-1]

    results = eval_retrieval_df.iloc[top_indices].copy()
    results["similarity"] = similarities[top_indices]

    return results.reset_index(drop=True)

In [ ]:
import pandas as pd

# Load our two baselines
baseline1 = pd.read_csv(
    "/content/baseline1_trivial_americanair.csv"
)

baseline2 = pd.read_csv(
    "/content/baseline2_tfidf_americanair.csv"
)

print("Baseline 1 rows:", len(baseline1))
print("Baseline 2 rows:", len(baseline2))

print("\nBaseline 2 columns:")
print(baseline2.columns.tolist())

In [ ]:
def deterministic_safety_gate(
    predicted_intent,
    retrieval_similarity,
    evidence_sufficient,
    confidence
):
    """
    Conservative safety gate.

    This is the deterministic layer of the final system.
    LLM judge will be added later.
    """

    high_risk_intents = {
        "accessibility",
        "lost_found",
        "payment_fees_refund",
        "cancellation_schedule",
        "rebooking_itinerary"
    }

    # 1. High-risk cases always escalate
    if predicted_intent in high_risk_intents:
        return "ESCALATE", "high_risk_intent"

    # 2. General / non-actionable cases should not be auto-resolved
    if predicted_intent == "general_complaint_or_other":
        return "ESCALATE", "general_or_underspecified"

    # 3. Insufficient evidence
    if not evidence_sufficient:
        return "ESCALATE", "insufficient_evidence"

    # 4. Low classifier confidence
    if confidence < 0.18:
        return "ESCALATE", "low_intent_confidence"

    # 5. Weak historical match
    if retrieval_similarity < 0.30:
        return "ESCALATE", "weak_historical_match"

    return "AUTO", "sufficient_evidence_and_confidence"

In [ ]:
def final_auto_escalate_decision(
    predicted_intent,
    retrieval_similarity,
    intent_confidence,
    judge_result
):
    """
    Final safety decision.

    AUTO is allowed only when:
    - intent is not high risk
    - intent confidence is adequate
    - historical evidence is strong
    - LLM judge finds the reply relevant and grounded
    - hallucination risk is low
    - judge explicitly considers it safe
    """

    high_risk_intents = {
        "accessibility",
        "lost_found",
        "payment_fees_refund",
        "cancellation_schedule",
        "rebooking_itinerary"
    }

    # --------------------------------------------------
    # 1. High-risk intents always require human review
    # --------------------------------------------------
    if predicted_intent in high_risk_intents:
        return "ESCALATE", "high_risk_intent"

    # --------------------------------------------------
    # 2. General / non-actionable messages
    # --------------------------------------------------
    if predicted_intent == "general_complaint_or_other":
        return "ESCALATE", "general_or_underspecified"

    # --------------------------------------------------
    # 3. Weak intent confidence
    # --------------------------------------------------
    if intent_confidence < 0.18:
        return "ESCALATE", "low_intent_confidence"

    # --------------------------------------------------
    # 4. Weak historical evidence
    # --------------------------------------------------
    if retrieval_similarity < 0.30:
        return "ESCALATE", "weak_historical_match"

    # --------------------------------------------------
    # 5. LLM judge evidence check
    # --------------------------------------------------
    if not judge_result.get("evidence_sufficient", False):
        return "ESCALATE", "judge_found_insufficient_evidence"

    # --------------------------------------------------
    # 6. LLM judge quality checks
    # --------------------------------------------------
    relevance = judge_result.get("relevance")
    grounding = judge_result.get("grounding")
    hallucination_risk = judge_result.get("hallucination_risk")

    if relevance is None or grounding is None or hallucination_risk is None:
        return "ESCALATE", "invalid_judge_result"

    if relevance < 4:
        return "ESCALATE", "low_reply_relevance"

    if grounding < 4:
        return "ESCALATE", "weak_reply_grounding"

    if hallucination_risk >= 3:
        return "ESCALATE", "hallucination_risk"

    if not judge_result.get("auto_safe", False):
        return "ESCALATE", "judge_not_safe_for_automation"

    # --------------------------------------------------
    # 7. Everything passed
    # --------------------------------------------------
    return "AUTO", "all_safety_checks_passed"

In [ ]:
def build_final_decision_record(
    row,
    predicted_intent,
    intent_confidence,
    retrieved_cases,
    generated_reply,
    judge_result
):
    """
    Combine retrieval, generation, judging and safety
    into one structured evaluation record.
    """

    top_similarity = float(
        retrieved_cases.iloc[0]["similarity"]
    )

    decision, reason = final_auto_escalate_decision(
        predicted_intent=predicted_intent,
        retrieval_similarity=top_similarity,
        intent_confidence=intent_confidence,
        judge_result=judge_result
    )

    return {
        "eval_id": row["eval_id"],
        "customer_text": row["customer_text"],
        "human_intent": row["human_intent"],
        "predicted_intent": predicted_intent,
        "intent_confidence": intent_confidence,
        "retrieval_similarity": top_similarity,
        "generated_reply": generated_reply,

        "judge_relevance": judge_result.get("relevance"),
        "judge_grounding": judge_result.get("grounding"),
        "judge_completeness": judge_result.get("completeness"),
        "judge_tone": judge_result.get("tone"),
        "judge_hallucination_risk": judge_result.get(
            "hallucination_risk"
        ),
        "judge_evidence_sufficient": judge_result.get(
            "evidence_sufficient"
        ),
        "judge_auto_safe": judge_result.get(
            "auto_safe"
        ),
        "judge_reason": judge_result.get("reason"),

        "final_decision": decision,
        "final_decision_reason": reason,

        "human_safe_to_auto_handle": row[
            "safe_to_auto_handle"
        ],
        "human_evidence_sufficient": row[
            "evidence_sufficient"
        ]
    }

In [ ]:
import re
import numpy as np
import pandas as pd


def normalize_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r"https?://\S+", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def keyword_overlap_score(reply, expected_points):
    """
    Simple lexical coverage of expected resolution points.
    Used only as an automated diagnostic, not as the final quality metric.
    """

    reply_text = normalize_text(reply)
    expected_text = normalize_text(expected_points)

    if not reply_text or not expected_text:
        return 0.0

    expected_tokens = set(expected_text.split())
    reply_tokens = set(reply_text.split())

    if not expected_tokens:
        return 0.0

    return len(expected_tokens & reply_tokens) / len(expected_tokens)

In [ ]:
def detect_urls(text):
    if pd.isna(text):
        return 0

    return len(
        re.findall(r"https?://\S+|www\.\S+", str(text))
    )


def detect_phone_numbers(text):
    if pd.isna(text):
        return 0

    pattern = r"(?:\+?1[\s.-]?)?\(?\d{3}\)?[\s.-]\d{3}[\s.-]\d{4}"

    return len(re.findall(pattern, str(text)))


def detect_unsupported_promises(text):
    """
    Basic diagnostic for phrases that may indicate
    unsupported guarantees/promises.
    """

    if pd.isna(text):
        return 0

    text = str(text).lower()

    risky_phrases = [
        "we guarantee",
        "you will receive",
        "we will refund",
        "you are guaranteed",
        "your refund is confirmed",
        "we have refunded",
        "we will compensate",
        "compensation is guaranteed"
    ]

    return sum(
        phrase in text
        for phrase in risky_phrases
    )

In [ ]:
def calculate_automatic_reply_metrics(df):

    result = df.copy()

    result["reply_point_coverage"] = result.apply(
        lambda row: keyword_overlap_score(
            row["generated_reply"],
            row["expected_reply_points"]
        ),
        axis=1
    )

    result["url_count"] = result["generated_reply"].apply(
        detect_urls
    )

    result["phone_number_count"] = result["generated_reply"].apply(
        detect_phone_numbers
    )

    result["unsupported_promise_count"] = result[
        "generated_reply"
    ].apply(
        detect_unsupported_promises
    )

    print("Evaluation cases:", len(result))
    print(
        "Non-empty replies:",
        result["generated_reply"]
        .fillna("")
        .str.strip()
        .ne("")
        .sum()
    )

    print(
        "Mean expected-point coverage:",
        round(result["reply_point_coverage"].mean(), 4)
    )

    print(
        "Replies containing URLs:",
        (result["url_count"] > 0).sum()
    )

    print(
        "Replies containing phone numbers:",
        (result["phone_number_count"] > 0).sum()
    )

    print(
        "Replies with possible unsupported promises:",
        (result["unsupported_promise_count"] > 0).sum()
    )

    return result

In [ ]:
# Load the final golden set fresh
golden_for_evidence = pd.read_csv(
    "/content/golden_eval_americanair_220_FINAL.csv"
)

print("Golden cases:", len(golden_for_evidence))

print("\nCurrent evidence labels:")
print(
    golden_for_evidence["evidence_sufficient"]
    .value_counts()
)

In [ ]:
# STEP 9 — Load completed evaluation artifacts
# The expensive/manual evaluation was completed separately and saved
# under evaluation/. This submission notebook loads those results rather
# than re-running manual annotation or LLM API calls.

from pathlib import Path
import pandas as pd

EVAL_DIR = Path('/content/hiver_submission/evaluation')

golden_eval = pd.read_csv(
    EVAL_DIR / 'golden_eval_americanair_220_FINAL.csv'
)

safety_results = pd.read_csv(
    EVAL_DIR / 'evidence_safety_gate_americanair.csv'
)

improved_safety_results = pd.read_csv(
    EVAL_DIR / 'improved_safety_gate_americanair.csv'
)

evidence_agreement = pd.read_csv(
    EVAL_DIR / 'evidence_judge_human_agreement_100.csv'
)

intent_results = pd.read_csv(
    EVAL_DIR / 'intent_classifier_eval_americanair.csv'
)

retrieval_results = pd.read_csv(
    EVAL_DIR / 'intent_aware_retrieval_americanair.csv'
)

print('Evaluation artifacts loaded successfully.')
print('Golden cases:', len(golden_eval))
print('Safety evaluations:', len(safety_results))
print('Independent evidence reviews:', len(evidence_agreement))
print('Intent evaluations:', len(intent_results))


In [ ]:
import pandas as pd
import os

files = {
    "Intent classifier":
        "/content/intent_classifier_eval_americanair.csv",

    "Intent-aware retrieval":
        "/content/intent_aware_retrieval_americanair.csv",

    "Safety gate v1":
        "/content/evidence_safety_gate_americanair.csv",

    "Safety gate v2":
        "/content/improved_safety_gate_americanair.csv",

    "Threshold calibration":
        "/content/safety_threshold_calibration_americanair.csv"
}

for name, path in files.items():

    print("\n" + "=" * 90)
    print(name)
    print("=" * 90)

    if not os.path.exists(path):
        print("FILE NOT FOUND")
        continue

    df = pd.read_csv(path)

    print("Rows:", len(df))
    print("Columns:")
    print(df.columns.tolist())

    print("\nFirst 3 rows:")
    display(df.head(3))

In [ ]:
import pandas as pd
import numpy as np

v1 = pd.read_csv(
    "/content/evidence_safety_gate_americanair.csv"
)

v2 = pd.read_csv(
    "/content/improved_safety_gate_americanair.csv"
)

def evaluate_safety(df, name):

    human_auto = df["human_decision"].eq("AUTO")
    system_auto = df["system_decision"].eq("AUTO")

    tp = (system_auto & human_auto).sum()
    fp = (system_auto & ~human_auto).sum()
    fn = (~system_auto & human_auto).sum()
    tn = (~system_auto & ~human_auto).sum()

    precision = (
        tp / (tp + fp)
        if tp + fp > 0 else 0
    )

    safe_coverage = (
        tp / human_auto.sum()
        if human_auto.sum() > 0 else 0
    )

    unsafe_rate = fp / len(df)

    escalation_recall = (
        tn / (~human_auto).sum()
        if (~human_auto).sum() > 0 else 0
    )

    f1 = (
        2 * precision * safe_coverage /
        (precision + safe_coverage)
        if precision + safe_coverage > 0 else 0
    )

    decision_accuracy = (
        (system_auto == human_auto).mean()
    )

    return {
        "System": name,
        "Cases": len(df),
        "AUTO": system_auto.sum(),
        "ESCALATE": (~system_auto).sum(),
        "TP": tp,
        "Unsafe AUTO": fp,
        "Missed AUTO": fn,
        "Correct ESCALATE": tn,
        "Automation precision": round(precision, 4),
        "Safe automation coverage": round(safe_coverage, 4),
        "Unsafe auto rate": round(unsafe_rate, 4),
        "Escalation recall": round(escalation_recall, 4),
        "Decision accuracy": round(decision_accuracy, 4),
        "Automation F1": round(f1, 4)
    }


final_safety_df = pd.DataFrame([
    evaluate_safety(v1, "Safety Gate v1"),
    evaluate_safety(v2, "Safety Gate v2")
])

display(final_safety_df)

In [ ]:
intent_df = pd.read_csv(
    "/content/intent_classifier_eval_americanair.csv"
)

retrieval_df = pd.read_csv(
    "/content/intent_aware_retrieval_americanair.csv"
)

# Intent accuracy
intent_accuracy = (
    intent_df["predicted_intent"] ==
    intent_df["human_intent"]
).mean()

# Number of errors
intent_errors = (
    intent_df["predicted_intent"] !=
    intent_df["human_intent"]
).sum()

# Mean confidence
mean_confidence = intent_df["intent_confidence"].mean()

# Retrieval similarity
mean_similarity = retrieval_df["retrieval_similarity"].mean()

# Retrieval result agrees with HUMAN intent
retrieved_human_match = (
    retrieval_df["retrieved_intent"] ==
    retrieval_df["human_intent"]
).mean()

print("FINAL INTENT / RETRIEVAL METRICS")
print("-" * 50)
print("Intent accuracy:",
      round(intent_accuracy * 100, 2), "%")

print("Intent errors:",
      intent_errors)

print("Mean intent confidence:",
      round(mean_confidence, 4))

print("Mean retrieval similarity:",
      round(mean_similarity, 4))

print("Retrieved intent = human intent:",
      round(retrieved_human_match * 100, 2), "%")

In [ ]:
import pandas as pd
import numpy as np

# Load final artifacts
intent_df = pd.read_csv(
    "/content/intent_classifier_eval_americanair.csv"
)

retrieval_df = pd.read_csv(
    "/content/intent_aware_retrieval_americanair.csv"
)

v1 = pd.read_csv(
    "/content/evidence_safety_gate_americanair.csv"
)

v2 = pd.read_csv(
    "/content/improved_safety_gate_americanair.csv"
)

baseline1 = pd.read_csv(
    "/content/baseline1_trivial_americanair.csv"
)

baseline2 = pd.read_csv(
    "/content/baseline2_tfidf_americanair.csv"
)

evidence_df = pd.read_csv(
    "/content/evidence_judge_human_agreement_100.csv"
)


# ------------------------------------------------------------
# Baseline metrics
# ------------------------------------------------------------

def baseline_metrics(df, name):

    human_auto = df["safe_to_auto_handle"].astype(bool)
    system_auto = df["predicted_auto"].eq("AUTO")

    return {
        "System": name,
        "Intent accuracy": np.nan,
        "Mean retrieval similarity": (
            df["retrieval_similarity"].mean()
            if "retrieval_similarity" in df.columns
            else np.nan
        ),
        "AUTO rate": system_auto.mean(),
        "Automation precision": 0.0,
        "Safe automation coverage": 0.0,
        "Unsafe auto rate": 0.0,
        "Escalation recall": 1.0
    }


# ------------------------------------------------------------
# Safety metrics
# ------------------------------------------------------------

def safety_metrics(df, name):

    human_auto = df["human_decision"].eq("AUTO")
    system_auto = df["system_decision"].eq("AUTO")

    tp = (system_auto & human_auto).sum()
    fp = (system_auto & ~human_auto).sum()
    fn = (~system_auto & human_auto).sum()
    tn = (~system_auto & ~human_auto).sum()

    precision = tp / (tp + fp) if tp + fp else 0
    coverage = tp / human_auto.sum() if human_auto.sum() else 0
    unsafe = fp / len(df)
    escalation = tn / (~human_auto).sum() if (~human_auto).sum() else 0

    return {
        "System": name,
        "Intent accuracy": (
            (df["predicted_intent"] == df["human_intent"]).mean()
        ),
        "Mean retrieval similarity": df["retrieval_similarity"].mean(),
        "AUTO rate": system_auto.mean(),
        "Automation precision": precision,
        "Safe automation coverage": coverage,
        "Unsafe auto rate": unsafe,
        "Escalation recall": escalation
    }


final_report_table = pd.DataFrame([
    baseline_metrics(
        baseline1,
        "Baseline 1 — Always escalate"
    ),

    baseline_metrics(
        baseline2,
        "Baseline 2 — TF-IDF retrieval"
    ),

    safety_metrics(
        v1,
        "Final candidate — Safety Gate v1"
    ),

    safety_metrics(
        v2,
        "Strict safety Gate v2"
    )
])

# Format percentages
percentage_columns = [
    "Intent accuracy",
    "AUTO rate",
    "Automation precision",
    "Safe automation coverage",
    "Unsafe auto rate",
    "Escalation recall"
]

for col in percentage_columns:
    final_report_table[col] = (
        final_report_table[col] * 100
    ).round(2)

final_report_table["Mean retrieval similarity"] = (
    final_report_table["Mean retrieval similarity"]
    .round(3)
)

print("FINAL REPORT COMPARISON")
display(final_report_table)

print("\nEvidence judge-human agreement:",
      round(evidence_df["agreement"].mean() * 100, 2),
      "%")

In [ ]:
import pandas as pd

v1 = pd.read_csv(
    "/content/evidence_safety_gate_americanair.csv"
)

retrieval = pd.read_csv(
    "/content/intent_aware_retrieval_americanair.csv"
)

# ------------------------------------------------------------
# 1. Intent classification failures
# ------------------------------------------------------------

intent_errors = v1[
    v1["predicted_intent"] != v1["human_intent"]
].copy()

print("=" * 90)
print("1. INTENT CLASSIFICATION ERRORS")
print("=" * 90)

print("Total intent errors:", len(intent_errors))

display(
    intent_errors[
        [
            "eval_id",
            "customer_text",
            "human_intent",
            "predicted_intent",
            "intent_confidence"
        ]
    ].head(10)
)


# ------------------------------------------------------------
# 2. Lowest retrieval similarity
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("2. WEAKEST RETRIEVAL MATCHES")
print("=" * 90)

weak_retrieval = retrieval.sort_values(
    "retrieval_similarity"
).head(10)

display(
    weak_retrieval[
        [
            "eval_id",
            "customer_text",
            "human_intent",
            "predicted_intent",
            "retrieval_similarity",
            "retrieved_customer_text",
            "retrieved_support_text"
        ]
    ]
)


# ------------------------------------------------------------
# 3. Unsafe AUTO decisions
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("3. UNSAFE AUTO DECISIONS")
print("=" * 90)

unsafe_auto = v1[
    (v1["system_decision"] == "AUTO") &
    (v1["human_decision"] == "ESCALATE")
].copy()

print("Unsafe AUTO cases:", len(unsafe_auto))

display(
    unsafe_auto[
        [
            "eval_id",
            "customer_text",
            "human_intent",
            "predicted_intent",
            "intent_confidence",
            "retrieval_similarity",
            "system_reason"
        ]
    ]
)


# ------------------------------------------------------------
# 4. System reason distribution
# ------------------------------------------------------------

print("\n" + "=" * 90)
print("4. ESCALATION REASON DISTRIBUTION")
print("=" * 90)

display(
    v1["system_reason"]
    .value_counts()
    .rename_axis("system_reason")
    .reset_index(name="count")
)

In [ ]:
failure_analysis = pd.DataFrame([
    {
        "failure_mode": "Intent overlap between related operational issues",
        "eval_id": 1,
        "example": "@AmericanAir my flight tmrw is booked for 7:45pm & w/o warning I’m being asked to check in for a 5:08pm flight. What am I supposed to do?",
        "hypothesis": "Related operational intents share vocabulary, making TF-IDF centroid routing prone to confusion."
    },
    {
        "failure_mode": "Complex multi-issue complaints",
        "eval_id": 212,
        "example": "@AmericanAir Check in online was down. Its been 2 hours waiting in line to check in and no staff is explaining anything Flight 908 boards in 1 hour and still unable to check our bags",
        "hypothesis": "Single-label classification loses secondary issues and urgency when several problems occur together."
    },
    {
        "failure_mode": "Semantic similarity does not guarantee actionable evidence",
        "eval_id": 179,
        "example": "@AmericanAir Okay. Thanks!",
        "hypothesis": "Lexical similarity can retrieve topically related cases without understanding conversational state."
    },
    {
        "failure_mode": "Weak evidence for high-risk or specialized cases",
        "eval_id": 9,
        "example": "@AmericanAir Been waiting 45 minutes at DCA gate 39 for wheelchair to bag claim",
        "hypothesis": "Sparse accessibility examples make nearest-neighbour evidence unreliable; specialized cases need stricter evidence requirements."
    },
    {
        "failure_mode": "Positive/social messages confuse issue-oriented taxonomy",
        "eval_id": 114,
        "example": "@AmericanAir Thank you to James Peterson, the gate agent at DCA, for his wonderful customer service. Happy Thanksgiving!",
        "hypothesis": "The taxonomy focuses on support issues while the source data also contains praise, acknowledgements, and social messages."
    }
])

failure_analysis.to_csv(
    "/content/top5_failure_modes_americanair.csv",
    index=False
)

print("Saved:")
print("/content/top5_failure_modes_americanair.csv")

display(failure_analysis)

In [ ]:
headline_caveat = """
What is misleading about my headline number?

The 81% evidence-judge/human agreement is useful, but it should not
be interpreted as end-to-end system accuracy.

First, the 100-case evidence review is a manually reviewed subset of
the 220-case golden evaluation set and was not sampled to represent
real-world customer-support traffic prevalence.

Second, the human labels themselves represent reviewer judgments about
whether historical evidence is sufficient; they are not an objective
ground truth for whether a response would ultimately resolve a
customer's issue.

Third, the LLM evidence judge is intentionally conservative. It marked
20% of the reviewed cases as evidence-sufficient, compared with 37%
for the independent human review. Therefore, the 81% agreement
reflects agreement with a human review process rather than factual
accuracy.

Finally, the system's 56.36% intent accuracy and 21.74% automation
precision on the 220-case evaluation should not be treated as
production estimates. The golden set was intentionally constructed
using intent/topic buckets, and the automation classes are imbalanced
(54 safe-to-auto-handle vs 166 escalation cases). A trivial
always-escalate policy therefore achieves 100% escalation recall and
75.45% raw decision accuracy without automating anything.

The results are best interpreted as diagnostic evidence about where
the current system fails: intent routing and evidence quality are the
main bottlenecks, while stricter safety gating can reduce unsafe
automation only by also reducing automation coverage.
"""

with open(
    "/content/headline_number_caveat.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write(headline_caveat)

print(headline_caveat)
print("\nSaved: /content/headline_number_caveat.txt")

In [ ]:
decision_log = pd.DataFrame([
    {
        "decision": "Selected AmericanAir as the target brand",
        "reason": "It provided a large support corpus with 36,598 support replies and high usable customer-to-support pairing coverage, while avoiding selection based only on raw volume."
    },
    {
        "decision": "Used customer-to-brand reply pairs rather than isolated tweets",
        "reason": "A support intent and its resolution are better represented by the customer message together with the corresponding brand response."
    },
    {
        "decision": "Used an 80/20 temporal split",
        "reason": "Historical evidence should precede evaluation cases; this reduces future-information leakage and better approximates deployment."
    },
    {
        "decision": "Built a 13-intent taxonomy",
        "reason": "The taxonomy groups recurring operational support problems while retaining specialized high-risk categories such as accessibility and lost/found."
    },
    {
        "decision": "Kept general/social messages as an explicit intent",
        "reason": "The Twitter dataset contains praise, acknowledgements, and conversational messages that do not correspond to operational support intents."
    },
    {
        "decision": "Used keyword/topic exploration before defining intents",
        "reason": "Clustering and topic prevalence were used as exploratory signals rather than treating unsupervised clusters as final business intents."
    },
    {
        "decision": "Used a trivial always-escalate baseline",
        "reason": "It establishes a safety floor and exposes how misleading raw decision accuracy can be under an imbalanced AUTO/ESCALATE distribution."
    },
    {
        "decision": "Used TF-IDF retrieval as the simple retrieval baseline",
        "reason": "It provides a reproducible lexical retrieval baseline before adding intent-aware routing."
    },
    {
        "decision": "Restricted intent-aware retrieval to the predicted intent",
        "reason": "The system should prefer historical examples from the detected support category, but this also makes retrieval dependent on classifier quality."
    },
    {
        "decision": "Required both intent confidence and retrieval similarity for automation",
        "reason": "A confident classifier alone is insufficient, and a strong lexical match alone can still be irrelevant to the customer's actual situation."
    },
    {
        "decision": "Escalated high-risk intents deterministically",
        "reason": "Accessibility, lost/found, financial/refund, cancellation, and rebooking cases can require case-specific handling and therefore need stricter safety controls."
    },
    {
        "decision": "Added an evidence-sufficiency judge",
        "reason": "Retrieval similarity is only a proxy for whether historical evidence can safely support a response, so evidence quality needs a separate evaluation."
    },
    {
        "decision": "Used an independent 100-case human evidence review",
        "reason": "The earlier evidence labels were coupled to automation decisions, so an independent review was created to avoid circular evaluation."
    },
    {
        "decision": "Used checkpointed batched LLM evaluation",
        "reason": "Batching reduces API calls and checkpointing prevents loss of completed evaluation results if an API limit is reached."
    },
    {
        "decision": "Did not use raw decision accuracy as the headline metric",
        "reason": "The golden set contains 54 AUTO-safe and 166 ESCALATE cases, so an always-escalate system achieves 75.45% raw accuracy without automating anything."
    }
])

decision_log.to_csv(
    "/content/decision_log_americanair.csv",
    index=False
)

print("Decision log saved.")
print("Number of decisions:", len(decision_log))

display(decision_log)

In [ ]:
one_week_plan = pd.DataFrame([
    {
        "Day": "Day 1",
        "Focus": "Improve intent taxonomy",
        "Action": "Split overlapping operational intents using clearer decision boundaries, especially cancellation vs delay, booking vs seat, and payment/refund vs operational issues.",
        "Expected impact": "Reduce systematic intent confusion."
    },
    {
        "Day": "Day 2",
        "Focus": "Add conversational-state handling",
        "Action": "Detect acknowledgements, praise, and social messages separately from support requests so messages such as 'Okay. Thanks!' are not treated as operational issues.",
        "Expected impact": "Reduce irrelevant retrieval and unnecessary automation."
    },
    {
        "Day": "Day 3",
        "Focus": "Improve retrieval",
        "Action": "Replace pure TF-IDF nearest-neighbour retrieval with hybrid lexical + semantic retrieval and rerank candidates using intent and issue-specific signals.",
        "Expected impact": "Improve evidence relevance for paraphrased and complex complaints."
    },
    {
        "Day": "Day 4",
        "Focus": "Handle multi-intent messages",
        "Action": "Add multi-intent/issue extraction and prioritize urgent or high-risk issues before generating a response.",
        "Expected impact": "Reduce failures on messages containing several simultaneous problems."
    },
    {
        "Day": "Day 5",
        "Focus": "Strengthen safety policy",
        "Action": "Use stricter evidence requirements for financial, accessibility, lost/found, cancellation, and rebooking cases and require stronger agreement between routing and evidence signals.",
        "Expected impact": "Reduce unsafe automation."
    },
    {
        "Day": "Day 6",
        "Focus": "Expand evaluation",
        "Action": "Create a larger stratified evaluation set with separate positive/social, ambiguous, high-risk, and multi-intent slices and add more human review.",
        "Expected impact": "Improve confidence in generalization."
    },
    {
        "Day": "Day 7",
        "Focus": "End-to-end validation",
        "Action": "Re-run baselines, intent classification, retrieval, response quality, evidence judging, and safety metrics on the new evaluation set.",
        "Expected impact": "Measure whether improvements actually increase safe automation rather than only improving one component."
    }
])

one_week_plan.to_csv(
    "/content/one_week_improvement_plan_americanair.csv",
    index=False
)

print("One-week improvement plan saved.")
display(one_week_plan)

## Evaluation Artifacts

Final evaluation outputs are available in the repository's `evaluation/` directory. See `README.md` for methodology, results, failure modes, decision log, and improvement plan.
